# Three-arm control experiment

Compare three policies on the same held-out batches of the test year:

- **O:** an in-regime oracle trained on interleaved batches from the test year.
- **S:** a historical policy trained on the same number of episodes as O.
- **B:** the full-history policy used in the main experiment.

O minus S measures the distribution effect at matched sample size; B minus S
measures the sample-size effect within the historical distribution. The oracle
is a diagnostic comparison and is not available at deployment.

Same-date contracts stay in the same batch. The default split is
`interleave2`; `interleave3` and `headtail60` are additional diagnostics.
Run `05_three_arm_all_regimes.ipynb` to execute this experiment for all four
regimes and produce the summary used in Table 2.


In [ ]:
# ══ E2-0 (rev. B). config + pipeline bootstrap: setup + H0 + G2 only ══
# The OOD arm is NOT retrained (its evaluation is read from sealed artefacts, hash-gated in E2-2),
# so H1's 1.5 h of GATE1 retraining is no longer part of this experiment.
E2_REGIME  = 'mix_test2022'    # crash_test2018 | crash_test2020 | mix_test2022 | mix_test2024
SPLIT_MODE = 'interleave2'     # 'interleave2' (primary) | 'interleave3' (embargo variant) | 'headtail60' (ablation)
E2_SEEDS   = [0, 1, 2, 3, 4]
NB_PATH    = None              # explicit path to the v12 notebook if auto-detection fails

need_pipeline = not (globals().get('PANEL_REGIME') == E2_REGIME
                     and 'pack_te' in globals() and 'EXP_DIR' in globals())
if need_pipeline:
    import json as _j, re as _re, time as _tm
    from pathlib import Path
    import os
    import sys
    import runpy


    def _find_repository_root():
        here = Path.cwd().resolve()
        for candidate in (here, *here.parents):
            if (candidate / 'ctc_runtime.py').is_file():
                return candidate
        raise FileNotFoundError('Start Jupyter from the conformal-trust-control repository.')

    KIT_ROOT = _find_repository_root()

    _KIT = KIT_ROOT
    NB_V12 = _KIT / 'notebooks' / '06_common_budget_setup.ipynb'
    NB_PORTABLE_SETUP = _KIT / 'notebooks' / '03_evaluate_policy_arms.ipynb'
    assert NB_V12.is_file(), f'sealed v1.2 source not found: {NB_V12}'
    assert NB_PORTABLE_SETUP.is_file(), f'portable setup source not found: {NB_PORTABLE_SETUP}'
    NB_PATH = NB_V12
    print('pipeline source of truth:', NB_PATH)
    _nbj = _j.load(open(NB_PATH))
    _cc = [''.join(c['source']) for c in _nbj['cells'] if c['cell_type'] == 'code']
    _setup_nbj = _j.load(open(NB_PORTABLE_SETUP))
    _setup_cc = [''.join(c['source']) for c in _setup_nbj['cells'] if c['cell_type'] == 'code']
    _portable_candidates = [s for s in _setup_cc
                            if '# CTC local runtime' in s]
    assert len(_portable_candidates) == 1, 'portable setup cell not found uniquely'
    _legacy_setup_indices = [i for i, s in enumerate(_cc)
                             if '# CTC local runtime' in s]
    assert len(_legacy_setup_indices) == 1, 'sealed v1.2 setup cell not found uniquely'
    _cc[_legacy_setup_indices[0]] = _portable_candidates[0]

    _h0pos = next(i for i, s in enumerate(_cc) if s.lstrip().startswith('# ── H0.'))
    def _cellsrc(m):
        h = [s for s in _cc if s.lstrip().startswith(m)]
        assert len(h) == 1, f'marker not unique/found: {m}'
        return h[0]
    _ip = get_ipython()
    def _rn(tag, src):
        t0 = _tm.time(); print(f'\n┏━━ E2 bootstrap ▶ {tag} ' + '━' * 30)
        r = _ip.run_cell(src)
        if not r.success:
            print(f'┗━━ ✗ {tag} FAILED'); r.raise_error()
        print(f'┗━━ ✓ {tag} done in {(_tm.time()-t0)/60:.1f} min')
    for _i, _s in enumerate(_cc[:_h0pos]):
        _rn(f'setup {_i+1}/{_h0pos}', _s)
    _h0src = _re.sub(r"(?m)^PANEL_REGIME\s*=\s*'[^']+'",
                     f"PANEL_REGIME = '{E2_REGIME}'", _cellsrc('# ── H0.'), count=1)
    assert f"PANEL_REGIME = '{E2_REGIME}'" in _h0src
    _rn('H0', _h0src)
    _rn('G2', _cellsrc('# ── G2.'))
assert PANEL_REGIME == E2_REGIME, 'pipeline context not ready'

# record the runtime — the oracle arm's training environment goes into the manifest
import platform as _pf, subprocess as _sp, numpy as _np2, torch as _th2
try:
    _cpu_model = [l for l in _sp.run(['lscpu'], capture_output=True, text=True).stdout.splitlines()
                  if 'Model name' in l][0].split(':', 1)[1].strip()
except Exception:
    _cpu_model = 'unknown'
E2_RUNTIME = {'python': _pf.python_version(), 'numpy': _np2.__version__,
              'torch': _th2.__version__, 'threads': int(_th2.get_num_threads()),
              'cpu_model': _cpu_model}
print(f'\nE2 ready on {PANEL_REGIME} | split={SPLIT_MODE} | seeds={E2_SEEDS}')
print('runtime:', E2_RUNTIME)
print('note: bitwise lineage is CPU-microarchitecture-sensitive; E2 therefore reads the OOD arm from')
print('      the SEALED arrays (hash-gated) and never mixes retrained-OOD numbers across runtimes.')

In [ ]:
# ── E2-1 (rev. C). within-year split + distribution-match audit ──
from scipy.stats import ks_2samp
_dstart = pd.to_datetime(EPS_TE['date_start']).dt.floor('D').to_numpy()
_udays  = np.sort(np.unique(_dstart))
BATCH   = np.searchsorted(_udays, _dstart)          # chronological 5-day batch index per episode

def e2_masks(mode):
    if mode == 'interleave2':
        te = (BATCH % 2 == 0); tr = ~te
    elif mode == 'interleave3':                      # 1-batch embargo: test / dropped / train, repeating
        te = (BATCH % 3 == 0); tr = (BATCH % 3 == 2)
    elif mode == 'headtail60':
        cut = _udays[int(0.6 * len(_udays))]
        tr = _dstart < cut; te = ~tr
    else:
        raise ValueError(mode)
    return tr, te

def _feats(eps_df, bs_pnl, spots):
    # note: the stored 'S-1' path is normalised (its first value is a constant), so a raw price-level
    # feature is degenerate (KS ≡ 0) and is excluded; ratio and P&L features are unaffected.
    iv = np.array([float(e['ep_step_df']['v'].iloc[0])    for _, e in eps_df.iterrows()])
    s0 = np.array([float(e['ep_step_df']['S-1'].iloc[0])  for _, e in eps_df.iterrows()])
    sT = np.array([float(e['ep_step_df']['S-1'].iloc[-1]) for _, e in eps_df.iterrows()])
    return {'entry_iv': iv, 'abs_5d_ret': np.abs(np.log(sT / s0)), 'bs_pnl_bp': bs_pnl / spots * 1e4}

F_TE = _feats(EPS_TE, BS_PNL_TE, SP_TE)
BS_PNL_TR = _episode_pnl(pack_tr, pack_tr['b_pos']).detach().cpu().numpy()
SP_TR = np.array([float(e['ep_step_df']['S-1'].iloc[0]) for _, e in EPS_TR.iterrows()])
F_TR = _feats(EPS_TR, BS_PNL_TR, SP_TR)

_arows = []
for _mode in ['interleave2', 'interleave3', 'headtail60']:
    _tr, _te = e2_masks(_mode)
    for _f in F_TE:
        _arows.append(dict(split=_mode, feature=_f,
                           ks=float(ks_2samp(F_TE[_f][_tr], F_TE[_f][_te]).statistic),
                           n_train=int(_tr.sum()), n_test=int(_te.sum())))
for _f in F_TE:                                      # the OOD yardstick: pre-test training years vs test year
    _arows.append(dict(split='OOD_reference', feature=_f,
                       ks=float(ks_2samp(F_TR[_f], F_TE[_f]).statistic),
                       n_train=len(EPS_TR), n_test=len(EPS_TE)))
AUDIT = pd.DataFrame(_arows)
_piv = AUDIT.pivot(index='feature', columns='split', values='ks').round(3)
print(f'[{PANEL_REGIME}] distribution-match audit (KS distance; smaller = better matched):')
print(_piv.to_string())
_r = _piv['interleave2'].mean() / _piv['OOD_reference'].mean()
print(f'\ninterleave2 internal divergence = {_r:.2f}× the OOD divergence '
      + ('→ "in-distribution" label supported' if _r < 0.5 else '→ CAUTION: within-year drift is large; flag this regime'))

TR_MASK, TE_MASK = e2_masks(SPLIT_MODE)
EPS_E2TR = EPS_TE[TR_MASK].reset_index(drop=True)
EPS_E2TE = EPS_TE[TE_MASK].reset_index(drop=True)
pack_e2tr, pack_e2te = pack_episodes(EPS_E2TR), pack_episodes(EPS_E2TE)
SP_E2TE, DATES_E2TE = SP_TE[TE_MASK], _ep_dates(EPS_E2TE)
print(f'\nsplit={SPLIT_MODE}: {int(TR_MASK.sum())} train eps / {int(TE_MASK.sum())} test eps '
      f'/ {int(BATCH.max()) + 1} batches')

In [ ]:
# ── E2-2 (rev. C). hash gates + train the oracle (O) and small-OOD (S) arms ──
import hashlib as _hh, json as _jm
def _e2md5(p):
    h = _hh.md5()
    with open(p, 'rb') as fh:
        for ch in iter(lambda: fh.read(1 << 20), b''): h.update(ch)
    return h.hexdigest()[:12]
def _e2phash(pol):
    return _hh.md5(b''.join(p.detach().cpu().numpy().tobytes() for p in pol.parameters())).hexdigest()[:12]

# GATE-E2a: arm B = the SEALED unified_v3 arrays (the exact arrays behind Table 7.1); md5-verified
_manref = f'{EXP_DIR}/controls_manifest_{PANEL_REGIME}.json'
_updp = _jm.load(open(_manref)).get('upstream_dp') if os.path.exists(_manref) else None
DP_OOD = {}
for k in KAPPAS:
    _f = f'{UNI_DIR}/dp_{PANEL_REGIME}_k{k}.npy'
    DP_OOD[k] = np.load(_f)
    if _updp is not None:
        assert _e2md5(_f) == _updp[f'k{k}'], f'GATE-E2a fail: dp_k{k} md5 ≠ manifest upstream_dp'
E2A_HASH_OK = _updp is not None
print('★ GATE-E2a PASS — arm B = sealed unified_v3 arrays'
      + (' (md5 ≡ regime manifest)' if E2A_HASH_OK else '  ⚠ regime manifest absent — hash check skipped'))

# arm O — oracle in-regime: matched distribution, small n
E2_POLS, DP_IND = {}, {}
for k in KAPPAS:
    _ps = []
    for sd in E2_SEEDS:
        _pol = train_residual(pack_e2tr, k, BEST_BETA_BASE[k], ALPHA, EPOCHS, LR, seed=sd)
        E2_POLS[(k, sd)] = _pol
        _ps.append(episode_dpnl_bp_lambda(pack_e2te, _pol, np.ones(len(EPS_E2TE)), SP_E2TE))
    DP_IND[k] = np.mean(_ps, axis=0)
    print(f'κ={k}: oracle arm O trained ({len(E2_SEEDS)} seeds) | held-out mean {DP_IND[k].mean():+.2f} bp')

# GATE-E2b: within-session bitwise determinism (mirrors GATE2's check)
_k0, _s0 = KAPPAS[0], E2_SEEDS[0]
assert _e2phash(E2_POLS[(_k0, _s0)]) == _e2phash(
    train_residual(pack_e2tr, _k0, BEST_BETA_BASE[_k0], ALPHA, EPOCHS, LR, seed=_s0)), 'GATE-E2b fail'
print('★ GATE-E2b PASS — small-arm training bitwise deterministic in this runtime')

# arm S — small-OOD (rev. C): SAME episode count as O, sampled from the historical training years.
# O − S isolates distribution at matched n; B − S isolates data quantity at matched distribution.
S_SEED = 20260719
_dstr = pd.to_datetime(EPS_TR['date_start']).dt.floor('D').to_numpy()
_udtr = np.sort(np.unique(_dstr)); BATCH_TR = np.searchsorted(_udtr, _dstr)
_rngS = np.random.default_rng(S_SEED)
_perm = _rngS.permutation(np.unique(BATCH_TR))
_take, _cum, _tgt = [], 0, int(TR_MASK.sum())
for _b in _perm:
    if _cum >= _tgt: break
    _take.append(int(_b)); _cum += int((BATCH_TR == _b).sum())
S_MASK = np.isin(BATCH_TR, np.array(_take))
EPS_S = EPS_TR[S_MASK].reset_index(drop=True)
pack_s = pack_episodes(EPS_S)
print(f'arm S: {len(EPS_S)} eps from {len(_take)} historical batches '
      f'(target n = {_tgt}; sample seed {S_SEED})')
DP_S = {}
for k in KAPPAS:
    _ps = []
    for sd in E2_SEEDS:
        _pol = train_residual(pack_s, k, BEST_BETA_BASE[k], ALPHA, EPOCHS, LR, seed=sd)
        _ps.append(episode_dpnl_bp_lambda(pack_e2te, _pol, np.ones(len(EPS_E2TE)), SP_E2TE))
    DP_S[k] = np.mean(_ps, axis=0)
    print(f'κ={k}: small-OOD arm S trained ({len(E2_SEEDS)} seeds) | held-out mean {DP_S[k].mean():+.2f} bp')

In [ ]:
# ── E2-3 (rev. C). three-arm comparison on identical held-out blocks + artefacts ──
_gfile = f'{EXP_DIR}/dpm_base.npz'
_g5 = np.load(_gfile) if os.path.exists(_gfile) else None
if _g5 is not None and os.path.exists(_manref):
    _mfiles = _jm.load(open(_manref)).get('files', {})
    if 'dpm_base.npz' in _mfiles:
        assert _e2md5(_gfile) == _mfiles['dpm_base.npz'], 'guarded-OOD artefact md5 ≠ regime manifest'
        print('✓ guarded-OOD column artefact md5 ≡ regime manifest')
if _g5 is None:
    print('note: dpm_base.npz not found — guarded-OOD column will be omitted')
_rows = []
for k in KAPPAS:
    A = DP_IND[k]                       # O: oracle in-regime, held-out blocks (λ=1)
    S = DP_S[k]                         # S: small-OOD, same blocks
    B = DP_OOD[k][TE_MASK]              # B: sealed big-OOD, same blocks
    mA, lA, hA, _  = full_stat_ci(A, DATES_E2TE, np.mean)
    mS, lS, hS, _  = full_stat_ci(S, DATES_E2TE, np.mean)
    mB, lB, hB, _  = full_stat_ci(B, DATES_E2TE, np.mean)
    g_net,  gl,  gh,  gp  = full_stat_ci(A - B, DATES_E2TE, np.mean)   # net (O − B)
    g_dist, dl2, dh2, dp2 = full_stat_ci(A - S, DATES_E2TE, np.mean)   # distribution effect, matched n
    g_data, dl3, dh3, dp3 = full_stat_ci(B - S, DATES_E2TE, np.mean)   # data-quantity effect, matched distribution
    row = dict(regime=PANEL_REGIME, kappa=k, split=SPLIT_MODE,
               n_train=int(TR_MASK.sum()), n_small=len(EPS_S), n_test=int(TE_MASK.sum()),
               indist_mean=mA, indist_lo=lA, indist_hi=hA, indist_tail=cvar5(A),
               smallood_mean=mS, smallood_lo=lS, smallood_hi=hS, smallood_tail=cvar5(S),
               ood_mean=mB, ood_lo=lB, ood_hi=hB, ood_tail=cvar5(B),
               gap=g_net, gap_lo=gl, gap_hi=gh, gap_p=gp,
               dist_effect=g_dist, dist_lo=dl2, dist_hi=dh2, dist_p=dp2,
               data_effect=g_data, data_lo=dl3, data_hi=dh3, data_p=dp3)
    if _g5 is not None:
        C = _g5[f'k{k}|crc_rwc_aci'][TE_MASK]        # sealed guarded big-OOD arm (c = 5 bp), same blocks
        mC, lC, hC, _ = full_stat_ci(C, DATES_E2TE, np.mean)
        row.update(guarded_ood_mean=mC, guarded_ood_lo=lC, guarded_ood_hi=hC,
                   guarded_ood_tail=cvar5(C),
                   guarded_retention_vs_ood=float(mC / mB) if abs(mB) > 1e-9 else float('nan'))
    _rows.append(row)
    print(f"κ={k}: O {mA:+.2f} | S {mS:+.2f} | B {mB:+.2f}  ||  "
          f"dist(O−S) {g_dist:+.2f} [{dl2:+.2f},{dh2:+.2f}] p={dp2:.3f} | "
          f"data(B−S) {g_data:+.2f} [{dl3:+.2f},{dh3:+.2f}] p={dp3:.3f} | net(O−B) {g_net:+.2f} p={gp:.3f}")
E2R = pd.DataFrame(_rows)
E2R.to_csv(f'{EXP_DIR}/e2_results_{PANEL_REGIME}.csv', index=False)
AUDIT.to_csv(f'{EXP_DIR}/e2_audit_{PANEL_REGIME}.csv', index=False)
import time as _tt
_man = {'created': _tt.strftime('%Y-%m-%d %H:%M:%S'), 'regime': PANEL_REGIME,
        'experiment': 'E2 in-distribution control, rev. C (three arms: oracle O / small-OOD S / sealed big-OOD B)',
        'split_mode': SPLIT_MODE, 'seeds': E2_SEEDS,
        'beta_star_reused': {str(k): float(BEST_BETA_BASE[k]) for k in KAPPAS},
        'n_batches': int(BATCH.max()) + 1, 'n_train': int(TR_MASK.sum()),
        'n_small': len(EPS_S), 'n_test': int(TE_MASK.sum()),
        's_arm': {'sample_seed': S_SEED, 'n_batches': len(_take)},
        'identification': 'dist_effect = O−S (distribution, matched n); data_effect = B−S '
                          '(quantity, matched distribution); gap = O−B (net)',
        'runtime': E2_RUNTIME,
        'ood_source': 'sealed unified_v3 dp arrays (bitwise lineage of Table 7.1); guarded column '
                      'from sealed dpm_base.npz',
        'cross_runtime_note': 'bitwise lineage is CPU-microarchitecture-sensitive; measured effect of '
                              'retraining on a different CPU: per-episode |Δ| up to ≈3 bp, mean edge '
                              'shift ≤0.1 bp (GATE1 divergence diagnostic, AMD EPYC 7B12, 2026-07). '
                              'O and S are trained in the runtime recorded here; ≤0.1 bp is the '
                              'comparability bound for small-arm-vs-sealed means.',
        'gates': {'e2a_sealed_hash_verified': bool(E2A_HASH_OK),
                  'e2b_training_determinism': True},
        'oracle_disclaimer': 'O is a diagnostic arm; trains on data unavailable before deployment; not deployable',
        'files': {f: _e2md5(f'{EXP_DIR}/{f}') for f in
                  [f'e2_results_{PANEL_REGIME}.csv', f'e2_audit_{PANEL_REGIME}.csv']}}
with open(f'{EXP_DIR}/e2_manifest_{PANEL_REGIME}.json', 'w') as fh:
    _jm.dump(_man, fh, ensure_ascii=False, indent=1)
print(f'\nsaved → e2_results / e2_audit / e2_manifest for {PANEL_REGIME}')
print('next: restart kernel, change E2_REGIME, rerun. After all four regimes, run the aggregation cell.')

In [ ]:
# ── E2-4 (rev. C). four-regime aggregation (run after all four regimes; needs only the environment cell) ──
import json as _jj, time as _tt
_REGS = ['crash_test2018', 'crash_test2020', 'mix_test2022', 'mix_test2024']
_srows = []
for _rg in _REGS:
    _f = f'{DRIVE_REPO}/checkpoints/conformal_controls_{_rg}/e2_results_{_rg}.csv'
    if not os.path.exists(_f):
        print(f'  {_rg}: no E2 results yet — skip'); continue
    _srows.append(pd.read_csv(_f))
if _srows:
    E2S = pd.concat(_srows, ignore_index=True)
    base_cols = ['regime', 'kappa', 'indist_mean', 'smallood_mean', 'ood_mean',
                 'dist_effect', 'dist_p', 'data_effect', 'data_p', 'gap', 'gap_p']
    cols = [c for c in base_cols if c in E2S.columns] \
           + [c for c in ['guarded_ood_mean', 'guarded_retention_vs_ood'] if c in E2S.columns]
    print(E2S[cols].round(2).to_string(index=False))
    _n = len(E2S)
    if 'dist_effect' in E2S.columns:
        _dpos = int((E2S['dist_effect'] > 0).sum()); _dsig = int((E2S['dist_p'] <= 0.05).sum())
        _qpos = int((E2S['data_effect'] > 0).sum()); _qsig = int((E2S['data_p'] <= 0.05).sum())
        print(f'\ndistribution effect (O−S) > 0 in {_dpos}/{_n} cells (sig {_dsig}) | '
              f'data-quantity effect (B−S) > 0 in {_qpos}/{_n} cells (sig {_qsig})')
    print(f'net (O−B) > 0 in {int((E2S["gap"] > 0).sum())}/{_n} cells')
    _out = f'{DRIVE_REPO}/checkpoints/e2_summary.csv'
    E2S.to_csv(_out, index=False)
    with open(f'{DRIVE_REPO}/checkpoints/e2_manifest_panel.json', 'w') as fh:
        _jj.dump({'created': _tt.strftime('%Y-%m-%d %H:%M:%S'),
                  'regimes_present': sorted(E2S['regime'].unique().tolist()), 'n_cells': _n,
                  'note': 'E2 rev. C (§7.1): three-arm in-distribution control. dist_effect = O−S '
                          '(matched n); data_effect = B−S (matched distribution); gap = O−B (net).'},
                 fh, indent=1)
    print('saved →', _out, '+ e2_manifest_panel.json')
else:
    print('nothing to aggregate yet.')